### ---------------------------------------------------------------------
### Importe
### ---------------------------------------------------------------------

In [16]:
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import os

### ---------------------------------------------------------------------
### Parameter
### ---------------------------------------------------------------------

In [42]:
BASE_CSV = Path("/home/samel/01. Projekte/01. Master/COMPARE_RST/Daten/Student_Performance_Behavior_Dataset/Students_Performance_Dataset.csv")
BASE_CONFIG = Path("/home/samel/01. Projekte/01. Master/COMPARE_RST/Auswertung/configs/students_behavior_mit_Cutoffs.json")
OUT_DIR = Path("noise_experiments")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NOISE_LEVELS = [0.0, 0.05, 0.10, 0.15, 0.20]  # 0–20%

# Welche Spalte wird manipuliert:
TARGET_COL = "Total_Score"   # "Grade"
os.getcwd()

'/home/samel/01. Projekte/01. Master/COMPARE_RST'

In [40]:
def create_noisy_csv(noise_level: float) -> Path:
    """Erzeuge eine Kopie der CSV mit zufälligem Noise in TARGET_COL."""
    df = pd.read_csv(BASE_CSV)

    if noise_level <= 0:
        out_csv = OUT_DIR / f"students_unbiased_noise_0.csv"
        df.to_csv(out_csv, index=False)
        return out_csv

    n = int(len(df) * noise_level)
    idx = np.random.choice(df.index, n, replace=False)

    if TARGET_COL == "Grade":
        # Grades zufällig permutieren
        possible = df[TARGET_COL].unique()
        df.loc[idx, TARGET_COL] = np.random.choice(possible, size=n)
    else:
        # Punkte leicht randomisieren, z.B. +-10 Punkte
        df.loc[idx, TARGET_COL] = df.loc[idx, TARGET_COL] + np.random.randint(-10, 11, size=n)
        # Auf sinnvollen Bereich beschränken
        df[TARGET_COL] = df[TARGET_COL].clip(lower=0)

    out_csv = OUT_DIR / f"students_unbiased_noise_{int(noise_level*100)}.csv"
    df.to_csv(out_csv, index=False)
    return out_csv
    

def create_config_for_noise(noise_level: float, noisy_csv: Path) -> Path:
    """Erzeuge eine Konfiguration, die auf die noisy-CSV zeigt."""
    with BASE_CONFIG.open("r", encoding="utf-8") as f:
        cfg = json.load(f)

    # WICHTIG: im student_two_datasets-Modus wird unbiased_csv verwendet!
    cfg["data"]["unbiased_csv"] = str(noisy_csv)
    
    cfg["output"]["dir"] = str(OUT_DIR / f"results_noise_{int(noise_level*100)}")

    out_cfg = OUT_DIR / f"config_noise_{int(noise_level*100)}.json"
    with out_cfg.open("w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

    return out_cfg


def run_config(config_path: Path):
    """Rufe deine Pipeline-Logik mit einer Konfiguration auf."""
    
    venv_python = ".venv/bin/python"  # Für Linux/MAC - Windows müsste anders aufgerufen werden

    # Umgebung des Subprozesses vorbereiten
    env = os.environ.copy()
    env.pop("MPLBACKEND", None)       # Jupyter-Backend entfernen

    cmd = [venv_python, "-m", "Auswertung", str(config_path)]
    subprocess.run(cmd, check=True, env=env)


def extract_coverage(config_path: Path) -> dict:
    """Liest Coverage aus dem entsprechenden Ergebnis-JSON für diesen Run."""
    with config_path.open("r", encoding="utf-8") as f:
        cfg = json.load(f)

    result_dir = Path(cfg["output"]["dir"])
    coverage_file = result_dir / "coverage.json"  # ggf. anpassen

    if not coverage_file.exists():
        return {"coverage_decision": None, "coverage_pass": None}

    with coverage_file.open("r", encoding="utf-8") as f:
        cov = json.load(f)

    # Beispiel: cov = {"decision": 0.8, "pass": 0.9}
    out = {
        "coverage_decision": cov.get("decision", None),
        "coverage_pass": cov.get("pass", None),
    }
    return out


def main():
    rows = []

    for nl in NOISE_LEVELS:
        print(f"=== Running noise_level={nl*100:.0f}% ===")
        noisy_csv = create_noisy_csv(nl)
        cfg = create_config_for_noise(nl, noisy_csv)
        run_config(cfg)
        cov = extract_coverage(cfg)

        rows.append({
            "noise_level": nl,
            "coverage_decision": cov["coverage_decision"],
            "coverage_pass": cov["coverage_pass"],
        })

    df_res = pd.DataFrame(rows)
    df_res.to_csv(OUT_DIR / "noise_results.csv", index=False)
    print("Fertig. Ergebnisse in:", OUT_DIR / "noise_results.csv")

In [43]:
if __name__ == "__main__":
    main()

=== Running noise_level=0% ===
{'mode': 'student_two_datasets', 'data': {'biased_csv': './Daten/Student_Performance_Behavior_Dataset/Students_Grading_Dataset_Biased.csv', 'unbiased_csv': 'noise_experiments/students_unbiased_noise_0.csv', 'file': './data.csv', 'file_type': 'csv', 'categorical_definite': [], 'categorical_optional': []}, 'decision_attribute': 'Grade', 'bins': 4, 'cutoffs': {'age': [20, 22], 'Sleep_Hours_per_Night': [6, 8], 'Total_Score': [60, 70, 80, 90]}, 'create_pass': True, 'pass_grades': ['A', 'B', 'C', 'D'], 'exclude_attrs': ['Student_ID', 'Email'], 'operations': {'reduct_grade': True, 'rules_grade': True, 'coverage_grade': True, 'reduct_pass': True, 'rules_pass': True, 'coverage_pass': True, 'indiscernibility_pass': False, 'reduct': True, 'rules': True, 'indiscernibility': False, 'coverage': True, 'plots': True}, 'output': {'dir': 'noise_experiments/results_noise_0', 'prefix': '', 'save_intermediate_disc': False, 'save_reducts': True, 'save_rules': True, 'save_indis